# PPO on SuperAutoStrat

The reusable pieces live in `agents/ppo/`:

| file | what |
|---|---|
| `config.py` | `PPOConfig` hyperparameters, `get_device()` |
| `network.py` | `ActorCritic`: shared trunk, policy logits + value head, masked sampling |
| `envs.py` | `make_vec_env`: N parallel envs (the actors) |
| `rollout.py` | `collect_rollout`: run pi_old for T steps |
| `buffer.py` | `RolloutBuffer`: (T x N) storage + minibatches |
| `advantages.py` | **`compute_gae` is yours to write** (`pytest tests/test_ppo.py` checks it) |
| `losses.py` | `ppo_clip_loss`; **`value_loss`, `entropy_bonus` are yours** |
| `tracking.py` | `EpisodeTracker` for logging |

In [15]:
%load_ext autoreload
%autoreload 2

import torch
from torch import nn

from agents.ppo import (
    ActorCritic, EpisodeTracker, PPOConfig, RolloutBuffer, collect_rollout, compute_gae,
    entropy_bonus, get_device, make_vec_env, ppo_clip_loss, value_loss,
)

cfg = PPOConfig()
device = get_device()
print(f"Using {device} device")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using cpu device


# Parallel actors

`N = cfg.n_envs` environments, each in its own process, stepped together. `obs` is `(N, n_states)` and `mask` is `(N, n_actions)`.

In [16]:
envs, obs, mask = make_vec_env(cfg.n_envs, async_envs=cfg.async_envs, seed=0)

n_states = envs.single_observation_space.shape[0]
n_actions = int(envs.single_action_space.n)
obs.shape, mask.shape

((8, 1146), (8, 132))

# Make the Policy Exist

In [17]:
model = ActorCritic(n_states, n_actions).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, eps=1e-5)

# Make the Surrogate Objectives Exist

Sanity check that `ppo_clip_loss` works as an ordinary loss for Adam: build a loss on a fake batch, backprop, step, and watch it go down. (Advantages are random here; real ones come from GAE.)

$$
L_{t}^{CLIP} (\theta) = \min (r_t(\theta)\cdot \hat{A}, \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon)\cdot \hat{A}) \\

L_{t}^{VF} = (V_{\theta} (s_t) - V_{t}^{targ})^2 \\

S[\pi_{\theta}](s_t) = \text{Entropy bonus} \\

L^{CLIP + VF + S} = \mathbb{\hat{E}_t} [L_{t}^{CLIP} -c_1 L_{t}^{VF} + c_2 S[\pi_{\theta}](s_t)] 
$$

In [ ]:
model = ActorCritic(n_states, n_actions).to(device)  # throw away the fake update
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, eps=1e-5)

# Run Proximal Policy Optimization

In [ ]:
buffer = RolloutBuffer(cfg.n_steps, cfg.n_envs, n_states, n_actions, device)
tracker = EpisodeTracker(cfg.n_envs)

for update in range(cfg.n_updates):
    # 1. Run pi_old in N envs for T steps
    obs, mask, last_value = collect_rollout(envs, model, buffer, obs, mask, device, tracker)

    # 2. Compute advantage estimates A_1..A_T
    buffer.advantages, buffer.returns = compute_gae(
        buffer.rewards, buffer.values, buffer.dones, last_value, cfg.gamma, cfg.gae_lambda
    )

    # 3. Optimise surrogate L wrt theta, K epochs, minibatch size M <= N*T
    for epoch in range(cfg.n_epochs):
        for mb in buffer.minibatches(cfg.minibatch_size):
            dist, new_values = model.distribution(mb["obs"], mb["masks"])   # one forward pass, both heads
            new_log_probs = dist.log_prob(mb["actions"])

            loss = (
                ppo_clip_loss(new_log_probs, mb["log_probs"], mb["advantages"], cfg.clip_eps)
                + cfg.value_coef * value_loss(new_values, mb["returns"])
                - cfg.entropy_coef * entropy_bonus(dist.entropy())
            )
            
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            # 4. Theta update
            optimizer.step()

    episodes = tracker.pop()
    if episodes:
        rets, lens = zip(*episodes)
        print(f"update {update}: {len(episodes)} episodes, "
              f"mean return {sum(rets)/len(rets):.2f}, mean length {sum(lens)/len(lens):.0f}")

update 0: 6 episodes, mean return -2.33, mean length 78
update 1: 10 episodes, mean return -1.10, mean length 126
update 2: 7 episodes, mean return 0.43, mean length 117


In [20]:
# envs.close()